In [ ]:

import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3" #due to the impossibility of changing args.device in SFTTrainer :(

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

from transformers import AutoModelForCausalLM

from peft import LoraConfig, get_peft_model, TaskType
from gptqmodel import GPTQModel
from gptqmodel.utils.backend import BACKEND

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lora_transfer_pruning").is_dir())
sys.path.insert(0, str(ROOT))


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO:datasets:PyTorch version 2.10.0+cu128 available.


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0+e355470
Transformers : 5.9.0
Torch        : 2.10.0+cu128
Triton       : 3.6.0


In [2]:
DEVICE = "cuda:0"  # physical cuda:3 after remapping
MODEL = "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

In [3]:
print(os.environ.get("HF_HUB_CACHE"))
os.environ["HF_HUB_OFFLINE"] = "1"


/glazkov-dev/.cache


In [4]:
quantized_model = GPTQModel.load(
    MODEL, 
    device=DEVICE, 
    backend=BACKEND.GPTQ_TORCH, 
    profile="low_memory",
)

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


from_quantized: adapter: None


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/revision/main "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

INFO  Patched transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeSparseMoeBlock -> defuser.modeling.unfused_moe.qwen3_moe.LinearQwen3MoeSparseMoeBlock


... (17 outputs left out)

INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}



INFO  Kernel: loaded -> `[TorchLinear]`                                        


In [5]:
quantized_model

Qwen3MoeQModel(
  (model): Qwen3MoeForCausalLM(
    (model): Qwen3MoeModel(
      (embed_tokens): Embedding(151936, 2048)
      (layers): ModuleList(
        (0-47): 48 x Qwen3MoeDecoderLayer(
          (self_attn): Qwen3MoeAttention(
            (q_proj): TorchLinear()
            (k_proj): TorchLinear()
            (v_proj): TorchLinear()
            (o_proj): TorchLinear()
            (q_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
          )
          (mlp): LinearQwen3MoeSparseMoeBlock(
            (gate): Qwen3MoeTopKRouter()
            (experts): ModuleList(
              (0-127): 128 x Qwen3MoeMLP(
                (gate_proj): TorchLinear()
                (up_proj): TorchLinear()
                (down_proj): TorchLinear()
                (act_fn): SiLUActivation()
              )
            )
          )
          (input_layernorm): Qwen3MoeRMSNorm((2048,), eps=1e-06)
          (post_attention_layernorm): Qwen3MoeRMSNorm

In [6]:
from itertools import chain
def get_model_size_mb(model):
    """Подсчитать примерный размер весов модели в МБ"""
    total_params = 0
    total_bytes = 0
    
    for param in chain(model.parameters(), model.buffers()):
        num_params = param.numel()
        param_bytes = num_params * param.element_size()
        total_params += num_params
        total_bytes += param_bytes
    
    size_mb = total_bytes / (1024 ** 2)
    
    return size_mb

get_model_size_mb(quantized_model)

16141.28955078125

In [7]:
quantized_model.model.model.layers[0].self_attn.q_proj.qweight.numel()

1048576

In [8]:
device = DEVICE  # логический CUDA-индекс внутри текущего процесса

print("Model tensors, GiB:", get_model_size_mb(quantized_model.model) / 1024)
print("PyTorch allocated, GiB:", torch.cuda.memory_allocated(device) / 2**30)
print("PyTorch reserved, GiB:", torch.cuda.memory_reserved(device) / 2**30)

Model tensors, GiB: 15.762978076934814
PyTorch allocated, GiB: 15.771748542785645
PyTorch reserved, GiB: 19.55078125


## LORA CONFIG

In [9]:
FRACTION_ATTN_LAYERS = [45, 46, 47]
FRACTION_MLP_LAYERS = [45, 46, 47] 

In [10]:
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.0, 
    bias="none",
    target_parameters = [], #important!
    #finds nn.Parameters, but after GPTQ they are modules!
    # target_parameters=[ f"model.layers.{layer}.mlp.experts.down_proj"
    #     for layer in FRACTION_MLP_LAYERS #layers_to_transform doesnt affect on parameters
    # ], #+ ["gate_up_proj"], #"gate_up_proj"
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    layers_to_transform = FRACTION_ATTN_LAYERS
)

In [11]:
print(config.target_modules)
print(config.target_parameters)

{'v_proj', 'q_proj', 'k_proj', 'down_proj', 'o_proj', 'gate_proj', 'up_proj'}
[]


In [12]:
from unittest.mock import patch

with patch( # PEFT dont know that expert's Parameters became independent Modules.
    "peft.utils.transformers_weight_conversion.convert_peft_config_for_transformers",
    return_value=None,
):
    model = get_peft_model(quantized_model, config) 
#can do it many times for different layers
model.print_trainable_parameters()
#warning predictable, because PEFT dont know that expert's Parameters became independent Modules.

trainable params: 52,740,096 || all params: 687,863,808 || trainable%: 7.6672


In [13]:
print(model.base_model.targeted_module_names)

['model.model.layers.45.self_attn.q_proj', 'model.model.layers.45.self_attn.k_proj', 'model.model.layers.45.self_attn.v_proj', 'model.model.layers.45.self_attn.o_proj', 'model.model.layers.45.mlp.experts.0.gate_proj', 'model.model.layers.45.mlp.experts.0.up_proj', 'model.model.layers.45.mlp.experts.0.down_proj', 'model.model.layers.45.mlp.experts.1.gate_proj', 'model.model.layers.45.mlp.experts.1.up_proj', 'model.model.layers.45.mlp.experts.1.down_proj', 'model.model.layers.45.mlp.experts.2.gate_proj', 'model.model.layers.45.mlp.experts.2.up_proj', 'model.model.layers.45.mlp.experts.2.down_proj', 'model.model.layers.45.mlp.experts.3.gate_proj', 'model.model.layers.45.mlp.experts.3.up_proj', 'model.model.layers.45.mlp.experts.3.down_proj', 'model.model.layers.45.mlp.experts.4.gate_proj', 'model.model.layers.45.mlp.experts.4.up_proj', 'model.model.layers.45.mlp.experts.4.down_proj', 'model.model.layers.45.mlp.experts.5.gate_proj', 'model.model.layers.45.mlp.experts.5.up_proj', 'model.mod

In [14]:
print(model.active_adapters)
print("attn 0 q proj: grad", model.model.model.model.layers[0].self_attn.q_proj.weight.requires_grad)
print("attn 47 q proj: grad", model.model.model.model.layers[47].self_attn.q_proj.weight.requires_grad)
print("attn 47 q proj lora A: grad", model.model.model.model.layers[47].self_attn.q_proj.lora_A['default'].weight.requires_grad)

['default']
attn 0 q proj: grad False
attn 47 q proj: grad False
attn 47 q proj lora A: grad True


In [15]:
print("Model tensors, GiB:", get_model_size_mb(model.base_model) / 1024)
print("PyTorch allocated, GiB:", torch.cuda.memory_allocated(device) / 2**30)
print("PyTorch reserved, GiB:", torch.cuda.memory_reserved(device) / 2**30)

Model tensors, GiB: 15.959450244903564
PyTorch allocated, GiB: 15.968220710754395
PyTorch reserved, GiB: 19.55078125


In [16]:
model.model.model.model.layers[0].self_attn.q_proj

TorchLinear()

In [17]:
model.model.model.model.layers[0].mlp

LinearQwen3MoeSparseMoeBlock(
  (gate): Qwen3MoeTopKRouter()
  (experts): ModuleList(
    (0-127): 128 x Qwen3MoeMLP(
      (gate_proj): TorchLinear()
      (up_proj): TorchLinear()
      (down_proj): TorchLinear()
      (act_fn): SiLUActivation()
    )
  )
)

In [18]:
model.model.model.model.layers[47].mlp

LinearQwen3MoeSparseMoeBlock(
  (gate): Qwen3MoeTopKRouter()
  (experts): ModuleList(
    (0-127): 128 x Qwen3MoeMLP(
      (gate_proj): lora.GPTQLoraLinear(
        (base_layer): TorchLinear()
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): Linear(in_features=2048, out_features=16, bias=False)
        )
        (lora_B): ModuleDict(
          (default): Linear(in_features=16, out_features=768, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
        (quant_linear_module): TorchLinear()
      )
      (up_proj): lora.GPTQLoraLinear(
        (base_layer): TorchLinear()
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): Linear(in_features=2048, out_features=16, bias=False)
        )
        (lora_B): ModuleDict(
          (defa

In [19]:
model.model.model.model.layers[47].mlp.experts

ModuleList(
  (0-127): 128 x Qwen3MoeMLP(
    (gate_proj): lora.GPTQLoraLinear(
      (base_layer): TorchLinear()
      (lora_dropout): ModuleDict(
        (default): Identity()
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=2048, out_features=16, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=16, out_features=768, bias=False)
      )
      (lora_embedding_A): ParameterDict()
      (lora_embedding_B): ParameterDict()
      (lora_magnitude_vector): ModuleDict()
      (quant_linear_module): TorchLinear()
    )
    (up_proj): lora.GPTQLoraLinear(
      (base_layer): TorchLinear()
      (lora_dropout): ModuleDict(
        (default): Identity()
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=2048, out_features=16, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=16, out_features=768, bias=False)
      )
      (lora_embedding_A): ParameterDict()
      (lora_embeddi

In [20]:
model.peft_config

{'default': LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'q_proj', 'k_proj', 'down_proj', 'o_proj', 'gate_proj', 'up_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=[45, 46, 47], layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=[], use_bdlora=None, arrow_config=N

## TRANSFORMERLENS BRIDGE

In [21]:
from transformer_lens.model_bridge.bridge import TransformerBridge


model_bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model.base_model.model.model,
    #dtype=torch.float16,
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:H

In [1]:
from transformers import Qwen3MoeConfig

In [22]:
model_bridge.blocks

ModuleList(
  (0): BlockBridge(
    (hook_in): HookPoint(name='blocks.0.hook_in')
    (hook_out): HookPoint(name='blocks.0.hook_out')
    (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
    (_original_component): Qwen3MoeDecoderLayer(
      (self_attn): PositionEmbeddingsAttentionBridge(
        (hook_in): HookPoint(name='blocks.0.attn.hook_in')
        (hook_out): HookPoint(name='blocks.0.attn.hook_out')
        (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
        (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
        (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
        (hook_result): HookPoint(name='blocks.0.attn.hook_result')
        (hook_attn_in): HookPoint(name='blocks.0.attn.hook_attn_in')
        (hook_q_input): HookPoint(name='blocks.0.attn.hook_q_input')
        (hook_k_input): HookPoint(name='blocks.0.attn.hook_k_input')
        (hook_v_input): HookPoint(name='blocks.0.attn.hook_v_input')
        (hook_r

In [23]:
model_bridge.blocks[47].attn.q_proj.active_adapters

['default']

In [28]:
model_bridge.blocks[47].attn.q

LinearBridge(2048 -> 4096, bias=False, original_component=GPTQLoraLinear)

In [27]:
model_bridge.blocks[47].attn.q.name

'q_proj'

In [26]:
model_bridge.blocks[47].attn.q.lora_A['default']

Linear(in_features=2048, out_features=16, bias=False)

In [30]:
model_bridge.blocks[47].attn.q.base_layer.weight

_LinearWeightMetadata(device=cuda:0, dtype=torch.float16, shape=(4096, 2048))